# Model II — classical encoder + one linear head

This notebook is the deliberately simple classical comparison for Model II. It has **no quantum circuit, no orbit projection, no classical orbit mixer, and no nonlinear classifier head**. All learned weights start from a fresh random initialization.

```text
.npy image [B,1,96,96]
  -> deterministic eight-view D4 lift
  -> deterministic 8-channel morphology bank
  -> one shared CompactOrbitEncoder [B,8,128]
  -> mean over the eight views [B,128]
  -> exactly one Linear(128,3) head
```

The only trainable modules are the existing 242,338-parameter MBConv encoder and the 387-parameter linear head, for **242,725 trainable parameters** total. D4 lifting, morphology, and view averaging are deterministic operations.


## 1. Imports and runtime paths

All machine-specific paths remain blank in Git. Set the environment variables or replace the empty strings only on the training machine.


In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

# Locate this repository without committing a machine-specific absolute path.
REPOSITORY_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "d4_orqb").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Run this notebook from inside the deeplense-quantum repository")
SOURCE_ROOT = REPOSITORY_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from d4_orqb.config import Config
from d4_orqb.data import (
    CachedNPYDataset,
    _require_disjoint_visible_content,
    build_loaders,
    make_loader,
    prepare_cache,
)
from d4_orqb.encoder import (
    CompactOrbitEncoder,
    MorphologyChannelBank,
    d4_transform,
    d4_views,
)

DEVELOPMENT_ROOT = os.environ.get("D4_ORQB_DEVELOPMENT_ROOT", "")
TEST_ROOT = os.environ.get("D4_ORQB_TEST_ROOT", "")
CACHE_ROOT = os.environ.get("D4_ORQB_CACHE_ROOT", "")
OUTPUT_DIR = os.environ.get("D4_ORQB_OUTPUT_DIR", "")

EPOCHS = 68
LEARNING_RATE = 4.5e-6
COSINE_FLOOR_LEARNING_RATE = 4.5e-7
WARMUP_EPOCHS = 5
SEED = 0
STAGE_NAME = "classical_encoder_linear_seed0_68ep"
CONFIRM_FINAL_TEST_EVALUATION = os.environ.get(
    "D4_ORQB_CONFIRM_FINAL_TEST_EVALUATION", "0"
).strip().lower() in {"1", "true", "yes"}
FINAL_TEST_ONLY = os.environ.get(
    "D4_ORQB_FINAL_TEST_ONLY", "0"
).strip().lower() in {"1", "true", "yes"}

print({
    "development_root_set": bool(DEVELOPMENT_ROOT.strip()),
    "test_root_set": bool(TEST_ROOT.strip()),
    "cache_root_set": bool(CACHE_ROOT.strip()),
    "output_dir_set": bool(OUTPUT_DIR.strip()),
    "epochs": EPOCHS,
    "peak_learning_rate": LEARNING_RATE,
    "final_test_only": FINAL_TEST_ONLY,
})


{'development_root_set': True, 'test_root_set': True, 'cache_root_set': True, 'output_dir_set': True, 'epochs': 68, 'peak_learning_rate': 4.5e-06, 'final_test_only': False}


## 2. Encoder and linear classifier

The eight-view mean makes the logits D4 invariant while keeping the learned model to the selected shared encoder and one linear layer.


In [2]:
class EncoderLinearClassifier(nn.Module):
    def __init__(self, num_classes: int = 3) -> None:
        super().__init__()
        self.morphology = MorphologyChannelBank(reference_pixels=96)
        self.encoder = CompactOrbitEncoder(
            input_channels=self.morphology.output_channels
        )
        self.head = nn.Linear(self.encoder.output_dim, num_classes)

    def forward(
        self, images: torch.Tensor, return_features: bool = False
    ):
        images = images.contiguous()
        views = d4_views(images)
        batch, group, channels, height, width = views.shape
        flat_views = views.reshape(batch * group, channels, height, width)
        encoded = self.encoder(self.morphology(flat_views))
        orbit_features = encoded.reshape(batch, group, -1)
        pooled_features = orbit_features.mean(dim=1)
        logits = self.head(pooled_features)
        if return_features:
            return logits, {
                "orbit_features": orbit_features,
                "pooled_features": pooled_features,
            }
        return logits

    def parameter_report(self) -> dict[str, int | str]:
        count = lambda module: sum(
            parameter.numel()
            for parameter in module.parameters()
            if parameter.requires_grad
        )
        return {
            "architecture": "CompactOrbitEncoder + Linear(128, 3)",
            "morphology": count(self.morphology),
            "encoder": count(self.encoder),
            "linear_head": count(self.head),
            "total": count(self),
        }


## 3. Data-free architecture check

This verifies the parameter count, output shapes, input/parameter gradients, and invariant logits before opening the dataset.


In [3]:
verification_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
verification_model = EncoderLinearClassifier().to(verification_device)
report = verification_model.parameter_report()
assert report == {
    "architecture": "CompactOrbitEncoder + Linear(128, 3)",
    "morphology": 0,
    "encoder": 242_338,
    "linear_head": 387,
    "total": 242_725,
}, report

probe = torch.rand(1, 1, 32, 32, device=verification_device)
probe.requires_grad_(True)
verification_model.train()
logits, features = verification_model(probe, return_features=True)
assert logits.shape == (1, 3)
assert features["orbit_features"].shape == (1, 8, 128)
assert features["pooled_features"].shape == (1, 128)
logits.square().mean().backward()
assert probe.grad is not None and torch.isfinite(probe.grad).all()
assert any(
    parameter.grad is not None and torch.isfinite(parameter.grad).all()
    for parameter in verification_model.encoder.parameters()
)
assert verification_model.head.weight.grad is not None

verification_model.eval()
with torch.no_grad():
    reference = verification_model(probe.detach())
    d4_errors = {
        f"r{rotation}s{reflected}": float(
            (verification_model(
                d4_transform(probe.detach(), rotation, reflected)
            ) - reference).abs().max()
        )
        for reflected in (0, 1)
        for rotation in range(4)
    }
assert max(d4_errors.values()) < 2e-4, d4_errors
print({**report, "max_d4_logit_error": max(d4_errors.values())})
del verification_model, probe, logits, features
if torch.cuda.is_available():
    torch.cuda.empty_cache()


{'architecture': 'CompactOrbitEncoder + Linear(128, 3)', 'morphology': 0, 'encoder': 242338, 'linear_head': 387, 'total': 242725, 'max_d4_logit_error': 2.9802322387695312e-08}


## 4. Fixed Model-II development split

Model II uses the same fixed, class-stratified 80/20 development split as the canonical notebook. The official test directory is not accessed here. A fresh output directory is mandatory.


In [4]:
def write_json_atomic(path: Path, value) -> None:
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}")
    temporary.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    os.replace(temporary, path)

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def validate_development_partition(
    cache_dir: Path,
    train_indices: np.ndarray,
    validation_indices: np.ndarray,
    split_path: Path,
) -> dict[str, int | str]:
    manifest_path = cache_dir / "manifest.csv"
    if not manifest_path.is_file() or not split_path.is_file():
        raise FileNotFoundError("Development manifest or split file is missing")
    with manifest_path.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    rows.sort(key=lambda row: int(row["index"]))
    if [int(row["index"]) for row in rows] != list(range(len(rows))):
        raise RuntimeError("Development manifest indices are not contiguous")
    digests = np.asarray([row["sha256_visible"] for row in rows], dtype=object)
    labels = np.asarray([int(row["label"]) for row in rows], dtype=np.int64)
    digest_labels: dict[str, set[int]] = {}
    for digest, label in zip(digests.tolist(), labels.tolist()):
        digest_labels.setdefault(str(digest), set()).add(int(label))
    cross_label = [digest for digest, values in digest_labels.items() if len(values) > 1]
    if cross_label:
        raise RuntimeError(
            f"Development data has {len(cross_label)} visible digest(s) across labels"
        )
    train_indices = np.asarray(train_indices, dtype=np.int64)
    validation_indices = np.asarray(validation_indices, dtype=np.int64)
    if (
        train_indices.size == 0
        or validation_indices.size == 0
        or train_indices.min() < 0
        or validation_indices.min() < 0
        or train_indices.max() >= len(rows)
        or validation_indices.max() >= len(rows)
    ):
        raise RuntimeError("Development split indices are empty or out of range")
    train_digests = set(digests[train_indices].tolist())
    validation_digests = set(digests[validation_indices].tolist())
    overlap = train_digests.intersection(validation_digests)
    if overlap:
        raise RuntimeError(
            f"Train/validation share {len(overlap)} model-visible digest(s)"
        )
    return {
        "development_manifest_sha256": sha256_file(manifest_path),
        "split_indices_sha256": sha256_file(split_path),
        "training_visible_digest_count": len(train_digests),
        "validation_visible_digest_count": len(validation_digests),
        "train_validation_visible_digest_overlap": 0,
    }

config = Config(
    dataset_id="model_ii",
    development_root=DEVELOPMENT_ROOT,
    validation_root="",
    cache_root=CACHE_ROOT,
    output_dir=OUTPUT_DIR,
    stage="pretrain",
    pretrain_epochs=EPOCHS,
    pretrain_patience=EPOCHS + 1,
    pretrain_seed=SEED,
    pretrain_learning_rate=LEARNING_RATE,
    pretrain_core_learning_rate=LEARNING_RATE,
)
config.validate()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: 68-epoch training on CPU will be slow; CUDA is recommended.")

run_contract = {
    "dataset_id": "model_ii",
    "architecture": "CompactOrbitEncoder + Linear(128, 3)",
    "initialization": "fresh_random",
    "epochs": EPOCHS,
    "peak_learning_rate": LEARNING_RATE,
    "cosine_floor_learning_rate": COSINE_FLOOR_LEARNING_RATE,
    "warmup_epochs": WARMUP_EPOCHS,
    "seed": SEED,
    "split_seed": config.split_seed,
    "validation_fraction": config.val_fraction,
    "test_used_for_selection": False,
}
contract_path = config.output_path / "classical_run_contract.json"
selected_checkpoint = config.output_path / STAGE_NAME / "best.pt"
summary_path = config.output_path / STAGE_NAME / "summary.json"
split_path = config.output_path / "split_indices.npz"
development_cache = config.cache_path / config.cache_key

if FINAL_TEST_ONLY:
    if not CONFIRM_FINAL_TEST_EVALUATION:
        raise ValueError(
            "FINAL_TEST_ONLY requires CONFIRM_FINAL_TEST_EVALUATION=True"
        )
    if (
        not contract_path.is_file()
        or not selected_checkpoint.is_file()
        or not summary_path.is_file()
    ):
        raise FileNotFoundError(
            "Completed run contract or validation-selected checkpoint is missing"
        )
    saved_contract = json.loads(contract_path.read_text())
    for key, expected in run_contract.items():
        if saved_contract.get(key) != expected:
            raise RuntimeError(f"Completed run contract mismatch for {key}")
    with np.load(split_path) as saved_split:
        saved_train_indices = saved_split["train"]
        saved_validation_indices = saved_split["val"]
    current_provenance = validate_development_partition(
        development_cache,
        saved_train_indices,
        saved_validation_indices,
        split_path,
    )
    for key, observed in current_provenance.items():
        if saved_contract.get(key) != observed:
            raise RuntimeError(f"Completed data provenance mismatch for {key}")
    saved_summary = json.loads(summary_path.read_text())
    if saved_summary.get("checkpoint_sha256") != sha256_file(selected_checkpoint):
        raise RuntimeError("Validation-selected checkpoint hash mismatch")
    run_contract = saved_contract
    loaders = None
    print("Opened completed run for final-test-only evaluation.")
else:
    if config.output_path.exists():
        raise FileExistsError(
            f"Use a fresh OUTPUT_DIR; already exists: {config.output_path}"
        )
    config.output_path.mkdir(parents=True)
    loaders = build_loaders(config, seed=SEED, device=device)
    run_contract.update(
        validate_development_partition(
            development_cache,
            loaders.train_indices,
            loaders.validation_indices,
            split_path,
        )
    )
    write_json_atomic(contract_path, run_contract)
    print({
        "device": str(device),
        "classes": loaders.class_names,
        "training_samples": len(loaders.train.dataset),
        "validation_samples": len(loaders.validation.dataset),
        "validation_policy": loaders.metadata["validation_mode"],
        "official_test_opened": False,
    })


CACHE_PROGRESS 384/89104


CACHE_PROGRESS 768/89104


CACHE_PROGRESS 1152/89104


CACHE_PROGRESS 1536/89104


CACHE_PROGRESS 1920/89104


...

CACHE_PROGRESS 88704/89104


CACHE_PROGRESS 89088/89104


CACHE_PROGRESS 89104/89104


CACHE_COMPLETE /mnt/run/cache/model_ii_96


{'device': 'cuda', 'classes': ['axion', 'cdm', 'no_sub'], 'training_samples': 71283, 'validation_samples': 17821, 'validation_policy': 'fixed_stratified_development_split', 'official_test_opened': False}


## 5. Metrics and the 68-epoch training engine

Training always completes all 68 epochs. `best.pt` is chosen only by development-validation balanced accuracy, then accuracy, macro F1, and negative log loss.


In [5]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")

def classification_metrics(labels: np.ndarray, logits: np.ndarray) -> dict:
    shifted = logits - logits.max(axis=1, keepdims=True)
    exponent = np.exp(shifted)
    probabilities = exponent / exponent.sum(axis=1, keepdims=True)
    predictions = probabilities.argmax(axis=1)
    matrix = np.zeros((3, 3), dtype=np.int64)
    np.add.at(matrix, (labels, predictions), 1)
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)
    recall = np.divide(
        np.diag(matrix), support, out=np.zeros(3), where=support > 0
    )
    precision = np.divide(
        np.diag(matrix), predicted, out=np.zeros(3), where=predicted > 0
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros(3),
        where=(precision + recall) > 0,
    )
    nll = -np.log(
        probabilities[np.arange(len(labels)), labels].clip(1e-12, 1.0)
    ).mean()
    return {
        "samples": int(len(labels)),
        "accuracy": float((predictions == labels).mean()),
        "balanced_accuracy": float(recall.mean()),
        "macro_f1": float(f1.mean()),
        "nll": float(nll),
        "confusion_matrix": matrix.tolist(),
    }

@torch.no_grad()
def evaluate(model: nn.Module, loader, device: torch.device):
    model.eval()
    labels_parts, logits_parts, index_parts = [], [], []
    for images, labels, indices in loader:
        images = images.to(device, non_blocking=True).contiguous(
            memory_format=torch.channels_last
        )
        with torch.autocast(
            device_type=device.type,
            dtype=torch.bfloat16,
            enabled=device.type == "cuda",
        ):
            logits = model(images)
        labels_parts.append(labels.numpy())
        logits_parts.append(logits.float().cpu().numpy())
        index_parts.append(indices.numpy())
    labels = np.concatenate(labels_parts)
    logits = np.concatenate(logits_parts)
    indices = np.concatenate(index_parts)
    return classification_metrics(labels, logits), labels, logits, indices

def save_checkpoint_atomic(path: Path, value) -> None:
    temporary = path.with_name(f".{path.name}.tmp-{os.getpid()}")
    torch.save(value, temporary)
    os.replace(temporary, path)

def train_classical_model(config: Config, loaders, device: torch.device):
    output_dir = config.output_path / STAGE_NAME
    if output_dir.exists():
        raise FileExistsError(f"Stage output already exists: {output_dir}")
    output_dir.mkdir(parents=True)
    seed_everything(SEED)
    model = EncoderLinearClassifier().to(
        device=device, memory_format=torch.channels_last
    )
    assert model.parameter_report()["total"] == 242_725
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=config.weight_decay
    )
    steps_per_epoch = len(loaders.train)
    total_steps = EPOCHS * steps_per_epoch
    warmup_steps = WARMUP_EPOCHS * steps_per_epoch
    minimum_ratio = COSINE_FLOOR_LEARNING_RATE / LEARNING_RATE

    def learning_rate_factor(step: int) -> float:
        if step < warmup_steps:
            progress = step / max(warmup_steps - 1, 1)
            return minimum_ratio + (1.0 - minimum_ratio) * progress
        decay_updates = total_steps - warmup_steps
        progress = (step - warmup_steps + 1) / max(decay_updates, 1)
        progress = min(max(progress, 0.0), 1.0)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return minimum_ratio + (1.0 - minimum_ratio) * cosine

    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, learning_rate_factor
    )
    history = []
    best_key = (-math.inf, -math.inf, -math.inf, -math.inf)
    best_epoch = -1
    run_start = time.time()

    for epoch in range(EPOCHS):
        model.train()
        loss_sum = 0.0
        correct = 0
        seen = 0
        first_update_learning_rate = None
        last_update_learning_rate = None
        for images, targets, _ in loaders.train:
            images = images.to(device, non_blocking=True).contiguous(
                memory_format=torch.channels_last
            )
            targets = targets.to(device, non_blocking=True)
            update_learning_rate = optimizer.param_groups[0]["lr"]
            if first_update_learning_rate is None:
                first_update_learning_rate = update_learning_rate
            last_update_learning_rate = update_learning_rate
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(
                device_type=device.type,
                dtype=torch.bfloat16,
                enabled=device.type == "cuda",
            ):
                logits = model(images)
                loss = F.cross_entropy(
                    logits, targets, label_smoothing=config.label_smoothing
                )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
            batch = targets.numel()
            seen += batch
            loss_sum += float(loss.detach()) * batch
            correct += int((logits.argmax(dim=1) == targets).sum())

        metrics, labels, validation_logits, indices = evaluate(
            model, loaders.validation, device
        )
        selection_key = (
            metrics["balanced_accuracy"],
            metrics["accuracy"],
            metrics["macro_f1"],
            -metrics["nll"],
        )
        record = {
            "epoch": epoch + 1,
            "train_loss": loss_sum / seen,
            "train_accuracy": correct / seen,
            "validation": metrics,
            "first_update_learning_rate": first_update_learning_rate,
            "last_update_learning_rate": last_update_learning_rate,
            "next_step_learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(record)
        write_json_atomic(output_dir / "history.json", history)
        save_checkpoint_atomic(
            output_dir / "last.pt",
            {"model": model.state_dict(), "epoch": epoch + 1, "record": record},
        )
        print(f"EPOCH {json.dumps(record, sort_keys=True)}", flush=True)
        if selection_key > best_key:
            best_key = selection_key
            best_epoch = epoch + 1
            save_checkpoint_atomic(
                output_dir / "best.pt",
                {"model": model.state_dict(), "epoch": best_epoch, "record": record},
            )
            np.savez_compressed(
                output_dir / "best_validation_predictions.npz",
                indices=indices, labels=labels, logits=validation_logits,
            )

    checkpoint_path = output_dir / "best.pt"
    checkpoint = torch.load(
        checkpoint_path, map_location=device, weights_only=False
    )
    model.load_state_dict(checkpoint["model"], strict=True)
    final_metrics, _, _, _ = evaluate(model, loaders.validation, device)
    summary = {
        "stage": STAGE_NAME,
        "epochs_completed": EPOCHS,
        "best_epoch": best_epoch,
        "validation": final_metrics,
        "parameters": model.parameter_report(),
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "peak_learning_rate": LEARNING_RATE,
        "cosine_floor_learning_rate": COSINE_FLOOR_LEARNING_RATE,
        "warmup_epochs": WARMUP_EPOCHS,
        "initialization": "fresh_random",
        "official_test_evaluated": False,
        "wall_seconds": time.time() - run_start,
    }
    write_json_atomic(output_dir / "summary.json", summary)
    (output_dir / "validation_metrics.md").write_text(
        "# Development-validation metrics\n\n"
        f"- Epochs completed: {EPOCHS}\n"
        f"- Selected epoch: {best_epoch}\n"
        f"- Accuracy: {final_metrics['accuracy']:.6f}\n"
        f"- Balanced accuracy: {final_metrics['balanced_accuracy']:.6f}\n"
        f"- Macro F1: {final_metrics['macro_f1']:.6f}\n"
        "- Official test evaluated during checkpoint selection: No.\n"
    )
    print(f"SUMMARY {json.dumps(summary, sort_keys=True)}", flush=True)
    return checkpoint_path, summary


## 6. Train once from scratch

This is one classical stage, not 18 epochs plus a second stage. The single encoder-linear model itself receives all 68 epochs.


In [6]:
if FINAL_TEST_ONLY:
    print("Training skipped: completed run opened for final-test-only evaluation.")
    run_summary = json.loads(
        (config.output_path / STAGE_NAME / "summary.json").read_text()
    )
else:
    selected_checkpoint, run_summary = train_classical_model(
        config, loaders, device
    )
print("Validation-selected checkpoint:", selected_checkpoint)
print("Official test has not been evaluated by this cell.")


EPOCH {"epoch": 1, "first_update_learning_rate": 4.5e-07, "last_update_learning_rate": 1.2576757532281204e-06, "next_step_learning_rate": 1.2605810616929698e-06, "train_accuracy": 0.3374437102815538, "train_loss": 1.1000026497323727, "validation": {"accuracy": 0.35581617193199033, "balanced_accuracy": 0.3539472382730237, "confusion_matrix": [[5828, 0, 151], [5573, 0, 379], [5377, 0, 513]], "macro_f1": 0.22006064473841588, "nll": 1.0918456315994263, "samples": 17821}}


EPOCH {"epoch": 2, "first_update_learning_rate": 1.2605810616929698e-06, "last_update_learning_rate": 2.0682568149210903e-06, "next_step_learning_rate": 2.0711621233859397e-06, "train_accuracy": 0.45916978802800107, "train_loss": 1.0845806917795644, "validation": {"accuracy": 0.5457606194938556, "balanced_accuracy": 0.5454009351419838, "confusion_matrix": [[4451, 961, 567], [1688, 2161, 2103], [1035, 1741, 3114]], "macro_f1": 0.5366423823824528, "nll": 1.0749810934066772, "samples": 17821}}


EPOCH {"epoch": 3, "first_update_learning_rate": 2.0711621233859397e-06, "last_update_learning_rate": 2.8788378766140604e-06, "next_step_learning_rate": 2.8817431850789097e-06, "train_accuracy": 0.5890745339000883, "train_loss": 1.057636615609005, "validation": {"accuracy": 0.6137141574546883, "balanced_accuracy": 0.6140924040238741, "confusion_matrix": [[4123, 911, 945], [193, 2363, 3396], [0, 1439, 4451]], "macro_f1": 0.6168079164690331, "nll": 1.0292800664901733, "samples": 17821}}


EPOCH {"epoch": 4, "first_update_learning_rate": 2.8817431850789097e-06, "last_update_learning_rate": 3.68941893830703e-06, "next_step_learning_rate": 3.6923242467718794e-06, "train_accuracy": 0.6210597197087665, "train_loss": 0.9758079224155385, "validation": {"accuracy": 0.6101790023006566, "balanced_accuracy": 0.6108092993109349, "confusion_matrix": [[4247, 800, 932], [133, 1707, 4112], [0, 970, 4920]], "macro_f1": 0.6009004410075368, "nll": 0.9003197550773621, "samples": 17821}}


EPOCH {"epoch": 5, "first_update_learning_rate": 3.6923242467718794e-06, "last_update_learning_rate": 4.5e-06, "next_step_learning_rate": 4.499999967655141e-06, "train_accuracy": 0.6598487717969221, "train_loss": 0.8316981314253182, "validation": {"accuracy": 0.6947421581280512, "balanced_accuracy": 0.6953729950222876, "confusion_matrix": [[4706, 942, 331], [222, 2276, 3454], [0, 491, 5399]], "macro_f1": 0.6834791922713038, "nll": 0.7524616122245789, "samples": 17821}}


...

EPOCH {"epoch": 66, "first_update_learning_rate": 4.7256366531941395e-07, "last_update_learning_rate": 4.600626798850621e-07, "next_step_learning_rate": 4.6002667501691936e-07, "train_accuracy": 0.9834462634849824, "train_loss": 0.1313885093005373, "validation": {"accuracy": 0.983165927837944, "balanced_accuracy": 0.9832378715288456, "confusion_matrix": [[5825, 153, 1], [100, 5806, 46], [0, 0, 5890]], "macro_f1": 0.9831952768023354, "nll": 0.07054238766431808, "samples": 17821}}


EPOCH {"epoch": 67, "first_update_learning_rate": 4.6002667501691936e-07, "last_update_learning_rate": 4.525172345316822e-07, "next_step_learning_rate": 4.5249922588387274e-07, "train_accuracy": 0.9836286351584529, "train_loss": 0.13102526591163546, "validation": {"accuracy": 0.9837831771505527, "balanced_accuracy": 0.9838569457707712, "confusion_matrix": [[5813, 166, 0], [89, 5829, 34], [0, 0, 5890]], "macro_f1": 0.9838230151603401, "nll": 0.07094134390354156, "samples": 17821}}


EPOCH {"epoch": 68, "first_update_learning_rate": 4.5249922588387274e-07, "last_update_learning_rate": 4.5e-07, "next_step_learning_rate": 4.5e-07, "train_accuracy": 0.9835725207973851, "train_loss": 0.1308140477202345, "validation": {"accuracy": 0.9832220414118176, "balanced_accuracy": 0.9832971628297608, "confusion_matrix": [[5812, 167, 0], [89, 5820, 43], [0, 0, 5890]], "macro_f1": 0.9832567420496211, "nll": 0.06955612450838089, "samples": 17821}}


SUMMARY {"best_epoch": 67, "checkpoint_sha256": "75247bbc49831df11d5f2c2a1a5a79d6d40f8dff2ff33b512593ccccc1d27134", "cosine_floor_learning_rate": 4.5e-07, "epochs_completed": 68, "initialization": "fresh_random", "official_test_evaluated": false, "parameters": {"architecture": "CompactOrbitEncoder + Linear(128, 3)", "encoder": 242338, "linear_head": 387, "morphology": 0, "total": 242725}, "peak_learning_rate": 4.5e-06, "stage": "classical_encoder_linear_seed0_68ep", "validation": {"accuracy": 0.9837831771505527, "balanced_accuracy": 0.9838569457707712, "confusion_matrix": [[5813, 166, 0], [89, 5829, 34], [0, 0, 5890]], "macro_f1": 0.9838230151603401, "nll": 0.07094134390354156, "samples": 17821}, "wall_seconds": 1550.6910750865936, "warmup_epochs": 5}


Validation-selected checkpoint: /mnt/run/outputs/model_ii_classical/classical_encoder_linear_seed0_68ep/best.pt
Official test has not been evaluated by this cell.


## 7. Review development validation

Validation selection chooses the strongest balanced-accuracy checkpoint without using the official test set.


In [7]:
validation_accuracy = run_summary["validation"]["accuracy"]
official_test_marker = config.output_path / STAGE_NAME / "official_test_metrics.json"
print(json.dumps(run_summary, indent=2, sort_keys=True))
print({
    "development_validation_accuracy": validation_accuracy,
    "validation_is_not_a_test_prediction": True,
    "official_test_evaluated": official_test_marker.exists(),
})


{
  "best_epoch": 67,
  "checkpoint_sha256": "75247bbc49831df11d5f2c2a1a5a79d6d40f8dff2ff33b512593ccccc1d27134",
  "cosine_floor_learning_rate": 4.5e-07,
  "epochs_completed": 68,
  "initialization": "fresh_random",
  "official_test_evaluated": false,
  "parameters": {
    "architecture": "CompactOrbitEncoder + Linear(128, 3)",
    "encoder": 242338,
    "linear_head": 387,
    "morphology": 0,
    "total": 242725
  },
  "peak_learning_rate": 4.5e-06,
  "stage": "classical_encoder_linear_seed0_68ep",
  "validation": {
    "accuracy": 0.9837831771505527,
    "balanced_accuracy": 0.9838569457707712,
    "confusion_matrix": [
      [
        5813,
        166,
        0
      ],
      [
        89,
        5829,
        34
      ],
      [
        0,
        0,
        5890
      ]
    ],
    "macro_f1": 0.9838230151603401,
    "nll": 0.07094134390354156,
    "samples": 17821
  },
  "wall_seconds": 1550.6910750865936,
  "warmup_epochs": 5
}
{'development_validation_accuracy': 0.9837831771

## 8. Explicit one-time official-test evaluation

Run this only after the architecture, learning rate, epoch budget, and validation-selected checkpoint are frozen. Set `CONFIRM_FINAL_TEST_EVALUATION = True` (or its environment variable) and provide a nonempty `TEST_ROOT`. The measured accuracy is reported without using it for tuning.


In [8]:
if not CONFIRM_FINAL_TEST_EVALUATION:
    print(
        "OFFICIAL TEST SKIPPED. Freeze the run, then explicitly enable "
        "CONFIRM_FINAL_TEST_EVALUATION and rerun this cell once."
    )
else:
    if not TEST_ROOT.strip():
        raise ValueError("Set a nonempty TEST_ROOT for final evaluation")
    marker_path = config.output_path / STAGE_NAME / "official_test_metrics.json"
    if marker_path.exists():
        raise FileExistsError(
            f"Official test was already evaluated for this run: {marker_path}"
        )
    saved_summary = json.loads(summary_path.read_text())
    checkpoint_sha256 = sha256_file(selected_checkpoint)
    if saved_summary.get("checkpoint_sha256") != checkpoint_sha256:
        raise RuntimeError("Validation-selected checkpoint hash mismatch")
    if not development_cache.is_dir():
        raise FileNotFoundError(
            "Development cache is missing; cannot verify test disjointness"
        )
    test_cache = config.cache_path / f"{config.cache_key}_official_test"
    test_metadata = prepare_cache(
        TEST_ROOT,
        test_cache,
        config.image_size,
        device,
        io_workers=config.io_workers,
        storage_dtype=np.float16,
    )
    _require_disjoint_visible_content(development_cache, test_cache)
    test_dataset = CachedNPYDataset(test_cache)
    test_loader = make_loader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        workers=config.workers,
        seed=SEED + 20_000,
    )
    model = EncoderLinearClassifier().to(
        device=device, memory_format=torch.channels_last
    )
    checkpoint = torch.load(
        selected_checkpoint, map_location=device, weights_only=False
    )
    model.load_state_dict(checkpoint["model"], strict=True)
    test_metrics, labels, logits, indices = evaluate(model, test_loader, device)
    final_result = {
        "evaluation": "separate_official_test",
        "selected_epoch": int(checkpoint["epoch"]),
        "metrics": test_metrics,
        "samples": int(test_metadata["samples"]),
        "checkpoint_sha256": checkpoint_sha256,
        "test_used_for_selection": False,
    }
    np.savez_compressed(
        config.output_path / STAGE_NAME / "official_test_predictions.npz",
        indices=indices, labels=labels, logits=logits,
    )
    write_json_atomic(marker_path, final_result)
    run_summary = dict(saved_summary)
    run_summary["official_test_evaluated"] = True
    run_summary["official_test_metrics_file"] = marker_path.name
    write_json_atomic(summary_path, run_summary)
    print(json.dumps(final_result, indent=2, sort_keys=True))


CACHE_PROGRESS 384/15000


CACHE_PROGRESS 768/15000


CACHE_PROGRESS 1152/15000


CACHE_PROGRESS 1536/15000


CACHE_PROGRESS 1920/15000


...

CACHE_PROGRESS 14592/15000


CACHE_PROGRESS 14976/15000


CACHE_PROGRESS 15000/15000


CACHE_COMPLETE /mnt/run/cache/model_ii_96_official_test


{
  "checkpoint_sha256": "75247bbc49831df11d5f2c2a1a5a79d6d40f8dff2ff33b512593ccccc1d27134",
  "evaluation": "separate_official_test",
  "metrics": {
    "accuracy": 0.9846,
    "balanced_accuracy": 0.9846,
    "confusion_matrix": [
      [
        4881,
        119,
        0
      ],
      [
        83,
        4888,
        29
      ],
      [
        0,
        0,
        5000
      ]
    ],
    "macro_f1": 0.9845838538775283,
    "nll": 0.0702153891324997,
    "samples": 15000
  },
  "samples": 15000,
  "selected_epoch": 67,
  "test_used_for_selection": false
}
